# Transform Circuits Data

1. Read bronze circuits table
2. Keep only the columns required for Analytics (drop url column)
3. Standardise column names using snake_case
4. Rename columns to make them more meaningful
5. Filter out rows where circuit_id is null
6. Remove duplicated values
7. Transform values of columns in Title Case
8. Write the transformted data to a silver table 

In [0]:
%run ../00-common/01.enviroment-config

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.circuits'
silver_table = f'{catalog_name}.{silver_schema}.circuits'

## Step 1 - Read bronze `circuits` table

In [0]:
circuits_df = spark.read.table(bronze_table)


In [0]:
display(circuits_df)

## Step 2 - Keep only the columns required for Analytics (drop url column)


In [0]:
from pyspark.sql import functions as F

In [0]:
circuits_df_selected = circuits_df.select(
    F.col('circuitId'),
    F.col('circuitName'),
    F.col('lat'),
    F.col('long'),
    F.col('locality'),
    F.col('country'),
    F.col('ingestion_timestamp'),
    F.col('source_file')
)

## Step 3 and 4 - Rename columns names


In [0]:
circuits_df_renamed = circuits_df_selected.withColumnsRenamed(
    {
        'circuitId': 'circuit_id',
        'circuitName': 'circuit_name',
        'lat': 'latitude',
        'long': 'longitude'
    }
)

## Step 5 - Filter out rows where circuit_id is null

In [0]:
circuits_not_null = circuits_df_renamed.filter(F.col('circuit_id').isNotNull())

In [0]:
display(circuits_not_null)

## Step 6 - Remove duplicated values

In [0]:
circuits_distinct_df = circuits_not_null.distinct()

In [0]:
display(circuits_distinct_df)

## Step 7 - Transform values of columns in Title Case

In [0]:
circuits_df_final = circuits_distinct_df.withColumns(
    {
        'circuit_name': F.initcap('circuit_name'),
        'locality': F.initcap('locality')

    }
)

display(circuits_df_final)

## Final Step - Write the transformted data to a silver table 

In [0]:
(
    circuits_df_final.write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
%sql
SELECT * FROM formula1.silver.circuits